In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Load dataset
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [ ]:
# Check first few rows
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
train_df.isnull().sum()

EDA


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Count plot of survived
sns.countplot(x='Survived', data=train_df)
plt.show() # Display count plot before other plots

In [ ]:
# Age distribution
sns.histplot(train_df['Age'], kde=True)
plt.show() # Display age distribution before other plots

In [ ]:
# Correlation heatmap
# Select only numeric columns for correlation calculation
numeric_train_df = train_df.select_dtypes(include=['number'])
sns.heatmap(numeric_train_df.corr(), annot=True, cmap='coolwarm')
plt.show()


Data Preprocessing

In [ ]:
#Handle missing values:
train_df['Age'].fillna(train_df['Age'].median(), inplace=True)

In [ ]:
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)

In [ ]:
## Drop unnecessary columns
train_df.drop(columns=['Cabin', 'Ticket', 'Name'], inplace=True, errors='ignore')  # optional

In [ ]:
#Encode categorical variables:
train_df = pd.get_dummies(train_df, columns=['Sex', 'Embarked'], drop_first=True)

In [ ]:
#Split features and target:
X = train_df.drop('Survived', axis=1)
y = train_df['Survived']


Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


Baseline Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)


In [ ]:
# Predict
y_pred = baseline_model.predict(X_val)

In [ ]:
# Metrics
print("Accuracy:", accuracy_score(y_val, y_pred))
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

Improved Model (Tree-based)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#Train
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


In [ ]:
#Predict
y_pred_rf = rf_model.predict(X_val)

In [ ]:
#Matrices
print("RF Accuracy:", accuracy_score(y_val, y_pred_rf))
print(confusion_matrix(y_val, y_pred_rf))

Feature Importance

In [ ]:
#Using SHAP
import shap

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_val)

shap.summary_plot(shap_values[:, :, 1], X_val)  # 1 = survived

In [ ]:
# Using scikit-learn
feat_importances = pd.Series(rf_model.feature_importances_, index=X.columns)
feat_importances.sort_values().plot(kind='barh')
plt.title("Feature Importance")
plt.show()


Visualize Performance

In [ ]:
#Confusion matrix
import seaborn as sns

cm = confusion_matrix(y_val, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
#ROC Curve
from sklearn.metrics import roc_curve, auc

y_prob = rf_model.predict_proba(X_val)[:,1]
fpr, tpr, _ = roc_curve(y_val, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve')
plt.legend()
plt.show()


In [ ]:
#Cross-validation for better generalization:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(rf_model, X, y, cv=5)
print("CV Accuracy:", scores.mean())
